In [ ]:
import sys
import numpy as np
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

X_tr_np, X_val_np, X_train_np_unused, X_test_np, y_tr_np, y_val_np, y_train_np_unused, y_test_np = load(norm='tanh_norm')
param_file = ROOT_DIR / "best_hyperparams.txt"
params = {}
with open(param_file) as f:
    for line in f:
        k, v = line.strip().split(":", 1)
        params[k] = v

In [ ]:
norm_type = params["norm"]
layers = eval(params["layers"])
lr = float(params["learning_rate"])
input_do = float(params["input_dropout"])
hidden_do = float(params["hidden_dropout"])
epochs = int(params["epochs"])
X_train_np = np.vstack([X_tr_np, X_val_np])
y_train_np = np.concatenate([y_tr_np, y_val_np])
X_train_norm = X_train_np.copy()
X_test_norm = X_test_np.copy()

In [ ]:
model = Sequential()
for i, units in enumerate(layers):
    if i == 0:
        model.add(Dense(units, input_shape=(X_train_norm.shape[1],),
                        activation='relu', kernel_initializer='he_normal'))
        if input_do > 0:
            model.add(Dropout(input_do))
    elif i == len(layers) - 1:
        model.add(Dense(units, activation='linear',
                        kernel_initializer='he_normal'))
    else:
        model.add(Dense(units, activation='relu',
                        kernel_initializer='he_normal'))
        if hidden_do > 0:
            model.add(Dropout(hidden_do))
model.compile(
    loss='mean_squared_error',
    optimizer=SGD(learning_rate=lr, momentum=0.5))
model.fit(
    X_train_norm, y_train_np,
    epochs=epochs, batch_size=64, shuffle=True, verbose=1
)

In [ ]:
test_loss = model.evaluate(X_test_norm, y_test_np, verbose=0)
print("Final Test MSE:", test_loss)